Now we will construct our binary at-risk response variable based on if a patient started new blood pressure, diabetes, or new cholesterol/lipid medication after NHT start. We will also clean up and standardize other various columns. Features with over 50% null values have been removed.

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
PROCESSED_FILE = os.path.join("..", "data", "processed", "cardio_onc_prostate_03cleaned.csv")
CLEANED_FILE = os.path.join("..", "data", "processed", "cardio_onc_prostate_04cleaned.csv")

df = pd.read_csv(PROCESSED_FILE)

## Change bp_meds_post, lipid_meds_post, dm_meds_post to be binary 0/1
These are the columns we will use to construct our target variable. Right now all of them are (0: None; 1: One; 2: More than two). Changing to (0: None, 1: One or more). Keep NAs as is.

In [3]:
cols = ['bp_meds_post', 'lipid_meds_post', 'dm_meds_post']

print("Before:")
for col in cols:
    print(df[col].value_counts(dropna=False))

df[cols] = df[cols].where(df[cols].isna(), (df[cols] != 0).astype(int))

print("After:")
for col in cols:
    print(df[col].value_counts(dropna=False))

Before:
bp_meds_post
0.0    165
NaN     37
1.0     25
2.0     11
3.0      1
Name: count, dtype: int64
lipid_meds_post
0.0    175
NaN     36
1.0     19
2.0      9
Name: count, dtype: int64
dm_meds_post
0.0    161
NaN     37
2.0     25
1.0     16
Name: count, dtype: int64
After:
bp_meds_post
0.0    165
NaN     37
1.0     37
Name: count, dtype: int64
lipid_meds_post
0.0    175
NaN     36
1.0     28
Name: count, dtype: int64
dm_meds_post
0.0    161
1.0     41
NaN     37
Name: count, dtype: int64


## Create at_risk binary response variable column
If all are NAN, outputs NAN. Otherwise, outputs 1 if any of the 3 columns are 1.

In [4]:
df['at_risk'] = df[['bp_meds_post','lipid_meds_post','dm_meds_post']].max(axis=1)

print(df['at_risk'].value_counts(dropna=False))

at_risk
0.0    124
1.0     79
NaN     36
Name: count, dtype: int64


## Standardize specific_nht_used
Eliminate case errors, spelling errors, standardize brand names, handle multiple NHTs.

In [5]:
print("Before:")
print(df['specific_nht_used'].value_counts(dropna=False))

# Normalize text
col = 'specific_nht_used'
df[col] = df[col].str.strip().str.lower()

# Mapping
mapping = {
    'abiraterone': 'Abiraterone',
    'abiaterone': 'Abiraterone',
    'zytiga': 'Abiraterone',
    'darolutamide': 'Darolutamide',
    'enzalutamide': 'Enzalutamide',
    'enzaltamide': 'Enzalutamide',
    'apalutamide': 'Apalutamide',
    'apalutamide to darolutamide': 'Apalutamide'
}

# Apply mapping
df[col] = df[col].replace(mapping)

# Check
print("After:")
print(df[col].value_counts(dropna=False))

Before:
specific_nht_used
Abiraterone                    97
Darolutamide                   85
Enzalutamide                   19
Apalutamide                    12
NaN                             6
darolutamide                    6
abiraterone                     5
enzalutamide                    3
apalutamide                     1
ENzalutamide                    1
Apalutamide to Darolutamide     1
Enzaltamide                     1
Abiaterone                      1
Zytiga                          1
Name: count, dtype: int64
After:
specific_nht_used
Abiraterone     104
Darolutamide     91
Enzalutamide     24
Apalutamide      14
NaN               6
Name: count, dtype: int64


## Standardize adt_agent
Eliminate case errors, spelling errors, standardize brand names, handle multiple ADTs.

In [6]:
print("Before:")
print(df['adt_agent'].value_counts(dropna=False))

col = 'adt_agent'

# Normalize
df[col] = df[col].str.strip().str.lower()

# Mapping
mapping = {
    # Orgovyx
    'orgovyx': 'Orgovyx',
    'orogovyx': 'Orgovyx',
    'orgovyx and nubeqa': 'Orgovyx',


    # Lupron
    'lupron': 'Lupron',
    'lupron depot': 'Lupron',
    'leupron': 'Lupron',
    'lupon': 'Lupron',
    'lpron': 'Lupron',

    # Bicalutamide / Casodex
    'bicalutamide': 'Bicalutamide',
    'casodex': 'Bicalutamide',

    # Firmagon
    'firmagon': 'Firmagon',

    # Multi / transitions / combos
    'firmagon to lupron': 'Lupron',
    'lupron + casodex to lupron': 'Lupron',
    'lupron, orgovyx (d/c)': 'Lupron',
    'firmagon to orgovyx': 'Orgovyx',
    'lupron, firmagon': 'Lupron',
    'bicalutamide + lupron': 'Lupron',
    'bicalutamide, then lupron': 'Lupron',
    'casodex to lpron': 'Lupron',
    'bicalutamide to lupron to orgovyx': 'Orgovyx',
    'lupron + bicalutamide': 'Lupron',
    'lupron to orgovyx': 'Orgovyx',
    'bicalutamide to lupron': 'Lupron',
    'casodex to lupron': 'Lupron',
    'casodex to firmagon': 'Firmagon'
}

# Apply mapping
df[col] = df[col].replace(mapping)

# Check
print("After:")
print(df[col].value_counts(dropna=False))

Before:
adt_agent
Orgovyx                              70
Lupron                               49
NaN                                  44
Bicalutamide                         24
Firmagon                             18
orgovyx                               5
bicalutamide                          3
Orogovyx                              3
Firmagon to Lupron                    2
Lupron + Casodex to Lupron            2
Lupron, Orgovyx (d/c)                 2
Firmagon to orgovyx                   1
Orgovyx and Nubeqa                    1
Lupron, Firmagon                      1
bicalutamide + Lupron                 1
Lupron Depot                          1
bicalutamide, then Lupron             1
Casodex to Lpron                      1
Leupron                               1
Bicalutamide to lupron to orgovyx     1
lupron + bicalutamide                 1
lupron to orgovyx                     1
Bicalutamide to lupron                1
Casodex to Lupron                     1
Casodex to Firmagon   

## Standardize other columns that use 0/1/2 instead of 0/1

In [7]:
# List of columns by type
# Columns where 1 = Yes, 2 = No
binary_1_yes = [
    'bb_prior', 'ace_arb_prior', 'hx_hld',
    'hx_high_tg', 'statin_prior', 'other_lipid_prior', 'lipid_panel_checked', 'hx_cad', 
    'hx_mi_stent', 'hx_chf', 'hx_arrhythmia', 'hx_carotid', 'hx_pad', 'hx_cva', 'hx_dm2',
    'glucose_over_200', 'asa_use', 'cards_prior',
    'cards_post', 'cards_referral', 'diet_counseling', 'exercise_counseling',
    'echo_ordered', 'ecg_done'
]

# Columns where 0 = No, 1+ = Yes
count_cols = [
    'hx_smoking', 'bp_meds_prior', 'bp_meds_post', 'dm_noninsulin',
    'dm_meds_post', 'lipid_meds_post', 'non_onc_provider'
]

# Columns where 1 = No, 2/3 = Yes / outside
flip_cols = ['has_pcp', 'has_pcp_duplicate']

# Columns where 1 = Yes, 2 = No, 0 = NAN
na_zero_cols = ['hx_htn', 'on_insulin', 'a1c_checked']

# --- Conversions ---

print("Before:")
for col in binary_1_yes + count_cols + flip_cols + na_zero_cols + ['family_hx_hd']:
    print(f"{col}:\n{df[col].value_counts(dropna=False)}\n")

# Standard yes/no (1 = yes)
for col in binary_1_yes:
    df[col] = df[col].where(df[col].isna(), (df[col] == 1).astype(int))

# Count-style
df[count_cols] = df[count_cols].where(
    df[count_cols].isna(),
    (df[count_cols] != 0).astype(int)
)

# Flipped encoding
for col in flip_cols:
    df[col] = df[col].where(df[col].isna(), (df[col] != 1).astype(int))

# Zeros to NAN
df[na_zero_cols] = df[na_zero_cols].replace({
    1: 1,
    2: 0,
    0: np.nan
})

# Special case: Family history (3 → NaN)
df['family_hx_hd'] = df['family_hx_hd'].replace({
    1: 1,
    2: 0,
    3: np.nan,
    0: np.nan
})

# Check
print("After:")
for col in binary_1_yes + count_cols + flip_cols + na_zero_cols + ['family_hx_hd']:
    print(f"{col}:\n{df[col].value_counts(dropna=False)}\n")

Before:
bb_prior:
bb_prior
2.0    152
1.0     51
NaN     36
Name: count, dtype: int64

ace_arb_prior:
ace_arb_prior
2.0    125
1.0     78
NaN     36
Name: count, dtype: int64

hx_hld:
hx_hld
1.0    107
2.0     96
NaN     36
Name: count, dtype: int64

hx_high_tg:
hx_high_tg
2.0    182
NaN     37
1.0     20
Name: count, dtype: int64

statin_prior:
statin_prior
1.0    110
2.0     93
NaN     36
Name: count, dtype: int64

other_lipid_prior:
other_lipid_prior
2.0    178
NaN     36
1.0     25
Name: count, dtype: int64

lipid_panel_checked:
lipid_panel_checked
2.0    153
1.0     50
NaN     36
Name: count, dtype: int64

hx_cad:
hx_cad
2.0    166
NaN     37
1.0     36
Name: count, dtype: int64

hx_mi_stent:
hx_mi_stent
2.0    182
NaN     36
1.0     21
Name: count, dtype: int64

hx_chf:
hx_chf
2.0    191
NaN     36
1.0     12
Name: count, dtype: int64

hx_arrhythmia:
hx_arrhythmia
2.0    178
NaN     36
1.0     25
Name: count, dtype: int64

hx_carotid:
hx_carotid
2.0    190
NaN     36
1.0     13
N

## Drop pcp duplicate column

In [8]:
df = df.drop(columns=['has_pcp_duplicate'])

## Convert ethnicity from integer to string

In [9]:
print("Before")
print(df['ethnicity'].value_counts(dropna=False))

ethnicity_map = {
    1: "Caucasian",
    2: "Hispanic",
    3: "Asian",
    4: "Black",
    5: "Other"
}

df['ethnicity'] = df['ethnicity'].map(ethnicity_map)

print("After")
print(df['ethnicity'].value_counts(dropna=False))

Before
ethnicity
1.0    116
NaN     32
3.0     28
5.0     24
2.0     22
4.0     17
Name: count, dtype: int64
After
ethnicity
Caucasian    116
NaN           32
Asian         28
Other         24
Hispanic      22
Black         17
Name: count, dtype: int64


# Additional Preprocessing
- type casting the columns
- median null filling

In [10]:
# --- Additional preprocessing (from collinearity analysis) ---

# Parse date columns and derive duration features
date_cols = ["nht_auth_date", "nht_start_date", "adt_start_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors = "coerce")

df["days_auth_to_start"] = (df["nht_start_date"] - df["nht_auth_date"]).dt.days
df["days_adt_to_nht"] = (df["nht_auth_date"] - df["adt_start_date"]).dt.days

# Coerce continuous columns to numeric
num_cols = ["bmi", "age", "sbp", "dbp", "ascvd_10yr"]
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors = "coerce")

# Cast binary columns to Int8
binary_cols = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done",
    "non_onc_provider", "at_risk",
]
for col in binary_cols:
    df[col] = pd.to_numeric(df[col], errors = "coerce").astype("Int8")

# Label-encode categorical columns (retain originals; add _enc suffix)
from sklearn.preprocessing import LabelEncoder


df["prescribing_provider"] = df["prescribing_provider"].str.strip().str.upper()

cat_cols = ["ethnicity", "specific_nht_used", "adt_agent", "prescribing_provider"]
le_map = {}
for col in cat_cols:
    le = LabelEncoder()
    non_null = df[col].notna()
    df.loc[non_null, col + "_enc"] = le.fit_transform(
        df.loc[non_null, col].astype(str)
    )
    df[col + "_enc"] = df[col + "_enc"].astype("Int16")
    le_map[col] = le

df["unique_patient_id"] = (
    pd.to_numeric(df["unique_patient_id"], errors = "coerce").astype("Int32")
)

print(df.dtypes)
print(f"\nShape: {df.shape}")
print(f"Missing values per column:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

unique_patient_id                    Int32
ethnicity                           object
nht_auth_date               datetime64[ns]
nht_start_date              datetime64[ns]
bmi                                float64
specific_nht_used                   object
age                                float64
adt_start_date              datetime64[ns]
adt_agent                           object
hx_smoking                            Int8
family_hx_hd                          Int8
hx_htn                                Int8
sbp                                float64
dbp                                float64
bp_meds_prior                         Int8
bb_prior                              Int8
ace_arb_prior                         Int8
bp_meds_post                       float64
has_pcp                               Int8
hx_hld                                Int8
hx_high_tg                            Int8
statin_prior                          Int8
other_lipid_prior                     Int8
lipid_meds_

## Median & Mode null filling

In [11]:
# Columns to exclude from filling
exclude_cols = ['bp_meds_post', 'lipid_meds_post', 'dm_meds_post', 'at_risk']

# Numeric columns (excluding specified ones)
for col in num_cols:
    if col not in exclude_cols:
        df[col] = df[col].fillna(df[col].median())

# Binary columns (excluding specified ones)
for col in binary_cols:
    if col not in exclude_cols:
        df[col] = df[col].fillna(df[col].mode()[0])

remove_cols = ['bp_meds_post', 'lipid_meds_post', 'dm_meds_post']

df = df.drop(columns=remove_cols, errors='ignore')

## Collinearity Pairings

- include after meeting clarifications

In [12]:
# # 1. diet_counseling + exercise_counseling → combine into single "lifestyle_counseling" flag
# #    (1 if either was done)
# df["lifestyle_counseling"] = (
#     (df["diet_counseling"].fillna(0) | df["exercise_counseling"].fillna(0))
#     .astype("Int8")
# )
# df.drop(columns = ["diet_counseling", "exercise_counseling"], inplace = True)
#
# # 2. hx_htn / bp_meds_prior / ace_arb_prior
# #    bp meds and ACE/ARB are largely redundant given HTN diagnosis → drop both med columns
# df.drop(columns = ["bp_meds_prior", "ace_arb_prior"], inplace = True)
#
# # 3. hx_hld / statin_prior
# #    statin use is redundant given hyperlipidemia diagnosis → drop statin
# df.drop(columns = ["statin_prior"], inplace = True)
#
# # 5. cards_prior / cards_post  →  kept separate (no change)
# # 6. hx_cad / cards_prior      →  kept separate (no change)
#
# print("Remaining columns:", df.columns.tolist())
# print(f"\nShape after collinearity resolution: {df.shape}")

# Save df

In [13]:
print(f"Shape: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"\nNumeric summary:")
df.describe()

Shape: (239, 53)

Data types:
unique_patient_id                    Int32
ethnicity                           object
nht_auth_date               datetime64[ns]
nht_start_date              datetime64[ns]
bmi                                float64
specific_nht_used                   object
age                                float64
adt_start_date              datetime64[ns]
adt_agent                           object
hx_smoking                            Int8
family_hx_hd                          Int8
hx_htn                                Int8
sbp                                float64
dbp                                float64
bp_meds_prior                         Int8
bb_prior                              Int8
ace_arb_prior                         Int8
has_pcp                               Int8
hx_hld                                Int8
hx_high_tg                            Int8
statin_prior                          Int8
other_lipid_prior                     Int8
lipid_panel_checked     

,unique_patient_id,nht_auth_date,nht_start_date,bmi,age,adt_start_date,hx_smoking,family_hx_hd,hx_htn,sbp,...,echo_ordered,ecg_done,non_onc_provider,at_risk,days_auth_to_start,days_adt_to_nht,ethnicity_enc,specific_nht_used_enc,adt_agent_enc,prescribing_provider_enc
count,239.0,234,210,239.000000,239.000000,193,239.0,239.0,239.0,239.000000,...,239.0,239.0,239.0,203.0,209.000000,192.000000,207.0,233.0,195.0,237.0
mean,120.0,2023-07-05 00:00:00,2023-08-29 08:00:00,27.646025,71.518828,2022-03-27 12:55:57.512953344,0.297071,0.154812,0.656904,129.962343,...,0.138075,0.100418,0.100418,0.389163,7.717703,509.213542,1.985507,1.150215,2.051282,27.21097
min,1.0,2020-12-18 00:00:00,2018-04-14 00:00:00,17.080000,48.000000,1905-07-08 00:00:00,0.0,0.0,0.0,96.000000,...,0.0,0.0,0.0,0.0,-1665.000000,-731.000000,0.0,0.0,0.0,0.0
25%,60.5,2022-08-09 06:00:00,2022-10-12 12:00:00,24.500000,67.000000,2022-03-30 00:00:00,0.0,0.0,0.0,126.500000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,2.0,0.0,2.0,24.0
50%,120.0,2023-05-29 00:00:00,2023-08-24 00:00:00,27.530000,71.000000,2023-02-22 00:00:00,0.0,0.0,1.0,129.000000,...,0.0,0.0,0.0,0.0,0.000000,28.000000,2.0,1.0,2.0,34.0
75%,179.5,2024-09-30 12:00:00,2024-10-29 18:00:00,29.655000,77.000000,2024-09-12 00:00:00,1.0,0.0,1.0,135.000000,...,0.0,0.0,0.0,1.0,13.000000,184.750000,2.0,2.0,3.0,34.0
max,239.0,2025-04-09 00:00:00,2025-04-09 00:00:00,57.660000,96.000000,2025-12-05 00:00:00,1.0,1.0,1.0,184.000000,...,1.0,1.0,1.0,1.0,505.000000,42759.000000,4.0,3.0,3.0,42.0
std,69.137544,NaN,NaN,4.775504,8.295910,NaN,0.457927,0.362484,0.47574,12.877030,...,0.345703,0.301188,0.301188,0.488766,185.058753,3146.876234,1.094906,1.109797,1.039149,10.550414


In [14]:
# Save cleaned dataframe
df.to_csv(CLEANED_FILE, index = False)
print(f"Cleaned dataset saved to: {CLEANED_FILE}")


Cleaned dataset saved to: ../data/processed/cardio_onc_prostate_04cleaned.csv
